In [2]:
import pandas as pd

# Charger le fichier
df = pd.read_csv("../data/raw_partner_headlines.csv", index_col=0)
print(f" {len(df)} lignes chargées")

 1845559 lignes chargées


In [3]:
# Aperçu des colonnes
print("Colonnes :", df.columns.tolist())
print("\nTypes :", df.dtypes)
print("\nAperçu :")
df.head()

Colonnes : ['headline', 'url', 'publisher', 'date', 'stock']

Types : headline     str
url          str
publisher    str
date         str
stock        str
dtype: object

Aperçu :


,headline,url,publisher,date,stock
2,Agilent Technologies Announces Pricing of $5……...,http://www.gurufocus.com/news/1153187/agilent-...,GuruFocus,2020-06-01 00:00:00,A
3,Agilent (A) Gears Up for Q2 Earnings: What's i...,http://www.zacks.com/stock/news/931205/agilent...,Zacks,2020-05-18 00:00:00,A
4,J.P. Morgan Asset Management Announces Liquida...,http://www.gurufocus.com/news/1138923/jp-morga...,GuruFocus,2020-05-15 00:00:00,A
5,"Pershing Square Capital Management, L.P. Buys ...",http://www.gurufocus.com/news/1138704/pershing...,GuruFocus,2020-05-15 00:00:00,A
6,Agilent Awards Trilogy Sciences with a Golden ...,http://www.gurufocus.com/news/1134012/agilent-...,GuruFocus,2020-05-12 00:00:00,A


In [4]:
# Voir les tickers disponibles
print("Nombre de tickers uniques :", df["stock"].nunique())
print("\nExemples de tickers :")
print(df["stock"].value_counts().head(20))

Nombre de tickers uniques : 6552

Exemples de tickers :
stock
KR      3314
GXC     3238
PGJ     3082
YINN    3027
JPM     2873
FXP     2854
XPP     2806
FCAU    2765
CHN     2698
JWN     2690
ERO     2669
RSP     2655
OXY     2647
DISH    2640
BLK     2570
VGK     2552
MDT     2545
UGAZ    2500
KEY     2492
AVGO    2488
Name: count, dtype: int64


In [5]:
# Vérifier si nos tickers existent
tickers = ["AAPL", "TSLA", "NVDA"]
for ticker in tickers:
    count = len(df[df["stock"] == ticker])
    print(f"{ticker} : {count} articles")

AAPL : 32 articles
TSLA : 0 articles
NVDA : 0 articles


In [7]:
import pandas as pd
import numpy as np
import random

random.seed(42)
np.random.seed(42)

tickers = ["AAPL", "TSLA", "NVDA"]
dates = pd.date_range(start="2020-01-01", end="2024-12-31", freq="B")

headlines_templates = {
    "AAPL": [
        "Apple reports record iPhone sales for Q{q}",
        "Apple launches new MacBook Pro with M{n} chip",
        "Apple stock hits all-time high amid strong earnings",
        "Apple faces antitrust scrutiny over App Store policies",
        "Apple announces ${n} billion share buyback program",
        "iPhone {n} pre-orders surpass expectations",
        "Apple expands into new markets with services growth",
        "Apple CEO Tim Cook discusses AI strategy for {y}",
    ],
    "TSLA": [
        "Tesla delivers record {n}K vehicles in Q{q}",
        "Tesla announces new Gigafactory in {city}",
        "Elon Musk unveils Tesla new battery technology",
        "Tesla stock surges after better-than-expected earnings",
        "Tesla faces production challenges at Berlin factory",
        "Tesla cuts prices amid growing EV competition",
        "Tesla FSD beta receives major update",
        "Tesla energy storage business hits new milestone",
    ],
    "NVDA": [
        "Nvidia reports record data center revenue of ${n}B",
        "Nvidia H{n} GPU demand surges amid AI boom",
        "Nvidia announces next-generation Blackwell architecture",
        "Nvidia stock soars after beating Wall Street estimates",
        "Nvidia partners with major cloud providers for AI chips",
        "Nvidia CEO Jensen Huang discusses AI future at CES",
        "Nvidia gaming GPU sales recover in Q{q}",
        "Nvidia expands AI chip production capacity",
    ]
}

cities = ["Texas", "Mexico", "India", "Germany"]
news_rows = []

for ticker in tickers:
    sampled_dates = np.random.choice(dates, size=1200, replace=False)
    templates = headlines_templates[ticker]
    
    for date in sampled_dates:
        date = pd.Timestamp(date)  # ← correction ici
        template = random.choice(templates)
        headline = template.format(
            q=random.randint(1, 4),
            n=random.randint(1, 100),
            y=date.year,
            city=random.choice(cities)
        )
        news_rows.append({
            "headline": headline,
            "date": date,
            "stock": ticker
        })

news_df = pd.DataFrame(news_rows)
news_df = news_df.sort_values("date").reset_index(drop=True)
news_df.to_csv("../data/news_clean.csv", index=False)
print(f"✅ news_clean.csv généré — {len(news_df)} articles")
print(news_df.groupby("stock").size())
print(news_df.head())

✅ news_clean.csv généré — 3600 articles
stock
AAPL    1200
NVDA    1200
TSLA    1200
dtype: int64
                                            headline       date stock
0  Tesla stock surges after better-than-expected ... 2020-01-01  TSLA
1  Nvidia partners with major cloud providers for... 2020-01-01  NVDA
2  Apple CEO Tim Cook discusses AI strategy for 2020 2020-01-01  AAPL
3               Tesla FSD beta receives major update 2020-01-02  TSLA
4  Nvidia partners with major cloud providers for... 2020-01-02  NVDA


In [1]:
import chromadb
from sentence_transformers import SentenceTransformer
import pandas as pd

# Charger les news
news_df = pd.read_csv("../data/news_clean.csv")
print(f"✅ {len(news_df)} articles chargés")

# Initialiser ChromaDB
client = chromadb.PersistentClient(path="../data/chroma_db")

# Créer la collection
collection = client.get_or_create_collection(
    name="financial_news",
    metadata={"hnsw:space": "cosine"}
)

print("✅ ChromaDB initialisé")

✅ 3600 articles chargés
✅ ChromaDB initialisé


In [2]:
from sentence_transformers import SentenceTransformer
import pandas as pd

# Charger le modèle d'embedding
print("⏳ Chargement du modèle all-MiniLM-L6-v2...")
model = SentenceTransformer('all-MiniLM-L6-v2')
print("✅ Modèle chargé")

# Peupler ChromaDB par batch
batch_size = 500
news_df = pd.read_csv("../data/news_clean.csv")

for i in range(0, len(news_df), batch_size):
    batch = news_df.iloc[i:i+batch_size]
    
    headlines = batch["headline"].tolist()
    embeddings = model.encode(headlines).tolist()
    ids = [f"news_{j}" for j in range(i, i+len(batch))]
    metadatas = [
        {"date": str(row["date"]), "stock": row["stock"]}
        for _, row in batch.iterrows()
    ]
    
    collection.add(
        documents=headlines,
        embeddings=embeddings,
        ids=ids,
        metadatas=metadatas
    )
    print(f"✅ Batch {i//batch_size + 1} ajouté — {i+len(batch)}/{len(news_df)}")

print(f"✅ ChromaDB peuplée — {collection.count()} documents")

⏳ Chargement du modèle all-MiniLM-L6-v2...


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


✅ Modèle chargé
✅ Batch 1 ajouté — 500/3600
✅ Batch 2 ajouté — 1000/3600
✅ Batch 3 ajouté — 1500/3600
✅ Batch 4 ajouté — 2000/3600
✅ Batch 5 ajouté — 2500/3600
✅ Batch 6 ajouté — 3000/3600
✅ Batch 7 ajouté — 3500/3600
✅ Batch 8 ajouté — 3600/3600
✅ ChromaDB peuplée — 3600 documents


In [3]:
# Test de recherche
query = "Apple earnings results"
results = collection.query(
    query_texts=[query],
    n_results=3
)

print(f"🔍 Recherche : '{query}'")
print("\nRésultats :")
for i, doc in enumerate(results["documents"][0]):
    meta = results["metadatas"][0][i]
    print(f"\n{i+1}. [{meta['stock']}] {meta['date']}")
    print(f"   {doc}")

C:\Users\pcc\.cache\chroma\onnx_models\all-MiniLM-L6-v2\onnx.tar.gz: 100%|██████████| 79.3M/79.3M [01:11<00:00, 1.16MiB/s]


🔍 Recherche : 'Apple earnings results'

Résultats :

1. [AAPL] 2020-12-22
   Apple stock hits all-time high amid strong earnings

2. [AAPL] 2021-01-26
   Apple stock hits all-time high amid strong earnings

3. [AAPL] 2022-01-12
   Apple stock hits all-time high amid strong earnings
